# A3 — Cross-Tool Severity Disagreement (Cohen's κ)

How far do the L2 tools disagree about how severe the same control is? Logic lives in `src/misconfig/disagreement.py` (CLAUDE.md rule 9).

**Three constraints shape this experiment, each a finding in its own right:**

1. **Checkov ships no severities** in its open-source repo (0 of 1,684 checks) — they live in the commercial Bridgecrew platform. It therefore cannot take part in a severity agreement statistic at all. Agreement is measured **Prowler vs ScoutSuite only**.
2. **The scales differ**: Prowler rates on five levels, ScoutSuite on two (`danger`/`warning`). κ over mismatched scales would measure the mapping as much as the tools, so the primary statistic collapses both to a shared binary scale (high-risk vs lower). The full 5×2 cross-tab is reported too.
3. **The tools share no identifiers**, so pairs come from semantic matching of check text, constrained to the same provider. The similarity threshold is a free parameter, so κ is reported across a **sweep** of thresholds rather than at one hand-picked value.

A note on what was *not* done: ScoutSuite ships a CIS 1.2.0 ruleset with recommendation numbers and Prowler ships CIS 1.4/1.5/2.0/3.0, but the sections were renumbered between those versions (CIS 1.2.0's §2.1 is §3.1 from 1.4 onward), so joining on recommendation numbers would silently produce wrong pairs.

In [1]:
import os
import sys
from pathlib import Path

# Every src/ path is relative to the project root (data/..., outputs/...).
# Without this chdir a run from notebooks/ writes into notebooks/data/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.misconfig.disagreement import run, log_stats

In [2]:
results = run()
log_stats(results)

2026-11-23 17:49:11,942 INFO src.misconfig.matching: Loading sentence-transformer model all-MiniLM-L6-v2


2026-11-23 17:49:11,970 INFO sentence_transformers.base.model: No device provided, using mps


2026-11-23 17:49:12,299 INFO httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-11-23 17:49:12,301 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-11-23 17:49:12,335 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"


2026-11-23 17:49:12,507 INFO httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


... (39 outputs left out)

/Users/spopa/Documents/Internship Project/internship-project/src/misconfig/matching.py:93: RuntimeWarning: divide by zero encountered in matmul
  similarity = emb_a @ emb_b.T
/Users/spopa/Documents/Internship Project/internship-project/src/misconfig/matching.py:93: RuntimeWarning: overflow encountered in matmul
  similarity = emb_a @ emb_b.T
/Users/spopa/Documents/Internship Project/internship-project/src/misconfig/matching.py:93: RuntimeWarning: invalid value encountered in matmul
  similarity = emb_a @ emb_b.T
2026-11-23 17:49:24,980 INFO src.misconfig.matching: Matched 61 prowler<->scoutsuite pairs at threshold 0.80


2026/11/23 17:49:26 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/spopa/Documents/Internship Project/internship-project/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026-11-23 17:49:26,809 INFO src.tracking: Logged A3 to MLflow (run f0b295d3d2284b259ce104db15ae3568)



A3 cross-tool severity disagreement

Severity availability per tool:
      tool  checks  with_severity  pct_with_severity  distinct_levels
   checkov    1684              0                0.0                0
   prowler    1608           1608              100.0                5
scoutsuite     578            531               91.9                2

Checkov is excluded from the agreement statistic: its open-source
repo ships no severities. Agreement is Prowler vs ScoutSuite only.

Kappa across similarity thresholds (binary scale: high-risk vs lower):
 threshold  n_pairs  n_rated  raw_agreement  kappa  ci_low  ci_high interpretation
      0.50      190      183          0.607  0.154   0.006    0.296         slight
      0.55      189      182          0.604  0.146   0.007    0.284         slight
      0.60      181      174          0.603  0.145   0.000    0.290         slight
      0.65      161      158          0.620  0.179   0.015    0.330         slight
      0.70      132      130 

## Threshold sensitivity

κ never leaves the "slight" band across the whole sweep, so the conclusion does not depend on the matcher's threshold.

In [3]:
results["sweep"]

,threshold,n_pairs,n_rated,raw_agreement,kappa,ci_low,ci_high,interpretation
0,0.50,190,183,0.607,0.154,0.006,0.296,slight
1,0.55,189,182,0.604,0.146,0.007,0.284,slight
2,0.60,181,174,0.603,0.145,0.000,0.290,slight
3,0.65,161,158,0.620,0.179,0.015,0.330,slight
4,0.70,132,130,0.646,0.219,0.038,0.392,fair
5,0.75,91,89,0.618,0.171,-0.045,0.367,slight
6,0.80,61,61,0.607,0.148,-0.101,0.390,slight


## Where the tools disagree

Inspect the matched pairs directly — the disagreements are systematic rather than random. CloudWatch alarm checks are the clearest case: Prowler rates every one `medium`, ScoutSuite rates every one `danger`.

In [4]:
results["by_service"].head(15)

,pairs,disagreements,disagreement_rate
prowler_service,,,
iam,18,7,0.389
cloudwatch,13,13,1.000
ec2,13,4,0.308
cloudsql,11,4,0.364
compute,9,2,0.222
sqlserver,8,3,0.375
logging,8,0,0.000
storage,7,3,0.429
postgresql,6,2,0.333


In [5]:
cols = [
    "similarity",
    "prowler_title",
    "scoutsuite_title",
    "prowler_severity_raw",
    "scoutsuite_severity_raw",
]
results["disagreements"][cols].head(20)

,similarity,prowler_title,scoutsuite_title,prowler_severity_raw,scoutsuite_severity_raw
1,0.889390,CloudWatch log metric filter and alarm exist f...,"No CloudWatch Alarm for ""Console Logins withou...",medium,danger
3,0.885068,Key Vault uses Azure RBAC for access control,Key Vault Role Based Access Control Disabled,high,warning
7,0.873586,CloudWatch Logs metric filter and alarm exist ...,"No CloudWatch Alarm for ""Security Group Changes""",medium,danger
9,0.870697,Key Vault has at least one diagnostic setting ...,Logging for Azure Key Vault Is Disabled,high,warning
11,0.858577,CloudWatch log metric filter and alarm exist f...,"No CloudWatch Alarm for ""S3 Bucket Policy Chan...",medium,danger
15,0.849998,Account monitors VPC route table changes with ...,"No CloudWatch Alarm for ""Route Table Changes""",medium,danger
16,0.846465,SQL server uses a customer-managed key for the...,SQL Server TDE Protector Not Encrypted with Cu...,critical,warning
17,0.846129,CloudWatch Logs metric filter and alarm exist ...,"No CloudWatch Alarm for ""AWS Configuration Cha...",medium,danger
19,0.844584,EBS default encryption is enabled,EBS Encryption By Default Is Disabled,high,warning
20,0.842313,Redshift cluster is not publicly exposed to th...,Cluster Publicly Accessible,critical,warning


## Match quality check

κ is only meaningful if the pairs really are the same control. Read the highest- and lowest-similarity pairs and judge them by eye before trusting the statistic.

In [6]:
pairs = results["pairs"]
display(pairs[cols].head(10))
display(pairs[cols].tail(10))

,similarity,prowler_title,scoutsuite_title,prowler_severity_raw,scoutsuite_severity_raw
0,0.890813,S3 bucket has server access logging enabled,Bucket Access Logging Disabled,medium,warning
1,0.889390,CloudWatch log metric filter and alarm exist f...,"No CloudWatch Alarm for ""Console Logins withou...",medium,danger
2,0.886368,Cloud SQL PostgreSQL instance has the log_min_...,Log Min Duration Statement Database Flag for P...,medium,warning
3,0.885068,Key Vault uses Azure RBAC for access control,Key Vault Role Based Access Control Disabled,high,warning
4,0.875525,GKE cluster does not use the Compute Engine de...,Default Service Account in Use,medium,warning
5,0.875400,Cloud Storage bucket has uniform bucket-level ...,Uniform Bucket-Level Access Is Disabled,medium,warning
6,0.874240,Cloud SQL PostgreSQL instance has log_connecti...,Log Connections Database Flag for PostgreSQL I...,medium,warning
7,0.873586,CloudWatch Logs metric filter and alarm exist ...,"No CloudWatch Alarm for ""Security Group Changes""",medium,danger
8,0.872726,Root account has MFA enabled,Root Account without MFA,critical,danger
9,0.870697,Key Vault has at least one diagnostic setting ...,Logging for Azure Key Vault Is Disabled,high,warning


,similarity,prowler_title,scoutsuite_title,prowler_severity_raw,scoutsuite_severity_raw
151,0.666759,IAM group does not have AdministratorAccess po...,Policy with Denied User Actions for Group Objects,high,danger
152,0.663937,Neptune cluster storage is encrypted at rest,Cluster Database Encryption Disabled,high,warning
153,0.662789,MySQL Flexible Server enforces SSL connections,Enforce SSL Connection Is Disabled for MySQL D...,high,warning
154,0.660617,Non-default network ACL is associated with a s...,Subnet with Default _ARG_0_ NACLs,low,warning
155,0.659555,Network ACL does not allow ingress from 0.0.0....,"Subnet with ""Allow All"" _ARG_0_ NACLs",high,warning
156,0.659011,ACM certificate is imported or has Certificate...,ACM Certificate with Transparency Logging Set ...,medium,warning
157,0.658481,Redshift cluster has automatic version upgrade...,Version Upgrade Disabled,medium,danger
158,0.656862,API server pod has --service-account-lookup se...,Automounted Service Account Token,high,danger
159,0.653496,Cluster role binding does not grant the cluste...,Permissive Bindings in Role-based Access Contr...,critical,warning
160,0.652140,Elastic Load Balancer has only HTTPS or SSL li...,Load Balancer Allowing Clear Text (HTTP) Commu...,medium,danger
